# hpyerparam tuning solvent


In [1]:
from model.models import MOFConditionPredictor
import torch
from torch_geometric.loader import DataLoader
from pathlib import Path
from fairmofsyncondition.crystal.crystal2data import DataStructure

import warnings

warnings.filterwarnings(
    "ignore",
    message="The usage of `scatter\\(reduce='max'\\).*",
    category=UserWarning,
)

%load_ext autoreload
%autoreload 2


IDX_TO_SOLVENT = {
    0: "WEVYAHXRMPXWCK-UHFFFAOYSA-N",
    1: "UHOVQNZJYSORNB-UHFFFAOYSA-N",
    2: "VLKZOEOYAKHREP-UHFFFAOYSA-N",
    3: "RTZKZFJDLAIYFH-UHFFFAOYSA-N",
    4: "JWUJQDFVADABEY-UHFFFAOYSA-N",
    5: "LYCAIKOWRPUZTN-UHFFFAOYSA-N",
    6: "XEKOWRVHYACXOJ-UHFFFAOYSA-N",
    7: "URLKBWYHVLBVBO-UHFFFAOYSA-N",
    8: "XLYOFNOQVPJJNP-UHFFFAOYSA-N",
    9: "OKKJLVBELUTLKV-UHFFFAOYSA-N",
    10: "XNLICIUVMPYHGG-UHFFFAOYSA-N",
    11: "YXFVVABEGXRONW-UHFFFAOYSA-N",
    12: "LRHPLDYGYMQRHN-UHFFFAOYSA-N",
    13: "LFQSCWFLJHTTHZ-UHFFFAOYSA-N",
    14: "ZSIAUFGUXNUGDI-UHFFFAOYSA-N",
    15: "ZMXDDKWLCZADIW-UHFFFAOYSA-N",
    16: "CSCPPACGZOOCGX-UHFFFAOYSA-N",
    17: "YMWUJEATGCHHMB-UHFFFAOYSA-N",
    18: "IAZDPXIOMUYVGZ-WFGJKAKNSA-N",
    19: "RYHBNJHYFVUHQT-UHFFFAOYSA-N",
    20: "HEDRZPFGACZZDS-UHFFFAOYSA-N",
    21: "IAZDPXIOMUYVGZ-UHFFFAOYSA-N",
    22: "WYURNTSHIVDZCO-UHFFFAOYSA-N",
}

/home/antonio/miniconda3/envs/dinga/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

cif_path = Path("../../../../Zn2C43N6H29O8.cif")
#cif_path = Path("../../../../EDUSIF.cif")
#cif_path = Path("../../../../ZnC5HO5.cif")

cif_path = Path("ADIQAG_fair.cif")
cif_path = Path("REWRAN_fair.cif")

data = DataStructure(str(cif_path)).complete_torch_data()

In [3]:
target_field, target_dim = "solvent", 23



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device= "cpu"
model = MOFConditionPredictor(
    target_dim=target_dim,
    global_fields=["lattice", "atomic_one_hot", "space_group_number" ]
)
model.load_state_dict(torch.load("best_model_solvent/best_model_solvent_seed_123.pt", map_location=device))



top_x = 3   

for batch in DataLoader([data], batch_size=1):
    break

model.eval()

with torch.no_grad():
    logits = model(batch)

    probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(
        probs,
        k=top_x,
        dim=-1
    )

for rank, (idx, prob) in enumerate(zip(top_indices[0], top_probs[0]), start=1):
    idx = int(idx.item())
    prob = float(prob.item())

    name = IDX_TO_SOLVENT[idx]

    print(f"{rank}. idx={idx} | prob={prob:.4f} | {name}")

1. idx=8 | prob=0.4728 | XLYOFNOQVPJJNP-UHFFFAOYSA-N
2. idx=9 | prob=0.2609 | OKKJLVBELUTLKV-UHFFFAOYSA-N
3. idx=13 | prob=0.1095 | LFQSCWFLJHTTHZ-UHFFFAOYSA-N
